In [24]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")

df = pd.read_csv(RAW_DIR / "online_retail_II.csv", encoding="ISO-8859-1")

print(df.shape)
df.head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 136.6 MB


In [26]:
# missing customer IDs
print("Missing Customer ID:", df["Customer ID"].isna().sum())

# cancelled orders (Invoice starting with 'C')
print("Cancelled invoices:", df["Invoice"].astype(str).str.startswith("C").sum())

# negative quantities (returns)
print("Negative quantity rows:", (df["Quantity"] < 0).sum())

# negative or zero prices
print("Non-positive price rows:", (df["Price"] <= 0).sum())

Missing Customer ID: 243007
Cancelled invoices: 19494
Negative quantity rows: 22950
Non-positive price rows: 6207


In [27]:
df_clean = df.copy()

# Drop missing Customer ID (non-negotiable for customer-level analysis)
df_clean = df_clean[df_clean["Customer ID"].notna()]

# Drop non-positive price (data artifacts, not real transactions)
df_clean = df_clean[df_clean["Price"] > 0]

# Flag cancellations instead of dropping them
df_clean["is_cancellation"] = df_clean["Invoice"].astype(str).str.startswith("C")

print(f"Rows before cleaning: {len(df)}")
print(f"Rows after cleaning:  {len(df_clean)}")
print(f"Dropped: {len(df) - len(df_clean)} ({(len(df)-len(df_clean))/len(df):.1%})")
print(f"\nCancellation rows retained: {df_clean['is_cancellation'].sum()}")
print(f"Purchase rows: {(~df_clean['is_cancellation']).sum()}")

Rows before cleaning: 1067371
Rows after cleaning:  824293
Dropped: 243078 (22.8%)

Cancellation rows retained: 18744
Purchase rows: 805549


In [ ]:
INTERIM_DIR = Path("../data/interim")
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

# parse InvoiceDate properly before saving, since we'll need real datetime math downstream
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])

df_clean.to_parquet(INTERIM_DIR / "retail_clean.parquet", index=False)
print(f"Saved {df_clean.shape} to {INTERIM_DIR / 'retail_clean.parquet'}")

Unique customer_id: 99441
Unique customer_unique_id: 96096


In [ ]:
check = pd.read_parquet(INTERIM_DIR / "retail_clean.parquet")
print(check.dtypes)
print("\nDate range:", check["InvoiceDate"].min(), "to", check["InvoiceDate"].max())
print("Unique customers:", check["Customer ID"].nunique())
print("Unique invoices:", check["Invoice"].nunique())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64